<a href="https://colab.research.google.com/github/momo4201/medical_imaging_and_informatics/blob/main/optuna_classification_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Radiomics Hub: https://radiomics.uk/

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Load CSV files
features_path = "/content/NSCLC-Radiomics_features.csv"
labels_path = "/content/NSCLC-Radiomics_labels.csv"

X_df = pd.read_csv(features_path)
y_df = pd.read_csv(labels_path)

print("Features shape:", X_df.shape)
print("Labels shape:", y_df.shape)


Features shape: (417, 110)
Labels shape: (422, 10)


In [ ]:
cols_to_drop = ["ROI", "img_path", "seg_path", "extraction_ID"]
X_df = X_df.drop(columns=cols_to_drop, errors="ignore")

In [ ]:
y_df['Histology'].value_counts()

,count
Histology,
squamous cell carcinoma,152
large cell,114
nos,63
adenocarcinoma,51


In [ ]:
id_col = "PatientID"

data = pd.merge(X_df, y_df[[id_col, "Histology"]], left_on="patient_ID", right_on="PatientID", how="inner")

print("Merged data shape:", data.shape)


Merged data shape: (417, 108)


In [ ]:
target_classes = ["adenocarcinoma", "squamous cell carcinoma"]

data = data[data["Histology"].str.lower().isin(target_classes)]

print(data["Histology"].value_counts())

Histology
squamous cell carcinoma    152
adenocarcinoma              50
Name: count, dtype: int64


In [ ]:
le = LabelEncoder()
data["label"] = le.fit_transform(data["Histology"])

print("Class mapping:")
for cls, enc in zip(le.classes_, range(len(le.classes_))):
    print(f"{cls} → {enc}")

Class mapping:
adenocarcinoma → 0
squamous cell carcinoma → 1


In [ ]:
X = data.drop(columns=[
    "patient_ID",
    "PatientID",
    "Histology",
    "label"
])

y = data["label"]

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("feature_selection", SelectKBest(score_func=f_classif)),
    ("classifier", RandomForestClassifier(random_state=42))
])


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)


Train size: (161, 105)
Test size: (41, 105)


In [ ]:
!pip install optuna

In [ ]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier


def objective(trial):

    # Feature selection
    k = trial.suggest_int("k", 10, min(200, X_train.shape[1]))

    # Model choice
    model_name = trial.suggest_categorical(
        "model", ["svm", "logistic", "decision_tree", "random_forest"]
    )

    if model_name == "svm":
        C = trial.suggest_float("svm_C", 1e-3, 1e2, log=True)
        gamma = trial.suggest_float("svm_gamma", 1e-4, 1e-1, log=True)
        model = SVC(
            C=C,
            gamma=gamma,
            kernel="rbf",
            probability=True,
            class_weight="balanced",
            random_state=42
        )

    elif model_name == "logistic":
        C = trial.suggest_float("logreg_C", 1e-3, 1e2, log=True)
        model = LogisticRegression(
            C=C,
            penalty="l2",
            solver="liblinear",
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )

    elif model_name == "decision_tree":
        max_depth = trial.suggest_int("dt_max_depth", 3, 30)
        min_samples_split = trial.suggest_int("dt_min_samples_split", 2, 10)
        min_samples_leaf = trial.suggest_int("dt_min_samples_leaf", 1, 5)
        model = DecisionTreeClassifier(
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            random_state=42
        )

    else:  # random_forest
        n_estimators = trial.suggest_int("rf_n_estimators", 100, 400)
        max_depth = trial.suggest_int("rf_max_depth", 5, 30)
        min_samples_split = trial.suggest_int("rf_min_samples_split", 2, 10)
        min_samples_leaf = trial.suggest_int("rf_min_samples_leaf", 1, 5)
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42
        )

    pipeline = Pipeline([
        ("scaler", StandardScaler()),
        ("selector", SelectKBest(score_func=f_classif, k=k)),
        ("clf", model)
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    aucs = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        pipeline.fit(X_tr, y_tr)
        y_prob = pipeline.predict_proba(X_val)[:, 1]
        aucs.append(roc_auc_score(y_val, y_prob))

    return np.mean(aucs)

In [ ]:
import optuna
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

print("Best ROC-AUC:", study.best_value)
print("Best configuration:")
print(study.best_params)

[I 2025-12-28 06:49:00,597] A new study created in memory with name: no-name-9e8404a7-c99e-47ba-b83e-9d9db212dca4
[I 2025-12-28 06:49:00,752] Trial 0 finished with value: 0.4468333333333333 and parameters: {'k': 37, 'model': 'svm', 'svm_C': 0.03607425403985285, 'svm_gamma': 0.0004783744934410723}. Best is trial 0 with value: 0.4468333333333333.
[I 2025-12-28 06:49:10,094] Trial 1 finished with value: 0.5475833333333333 and parameters: {'k': 59, 'model': 'random_forest', 'rf_n_estimators': 394, 'rf_max_depth': 19, 'rf_min_samples_split': 7, 'rf_min_samples_leaf': 4}. Best is trial 1 with value: 0.5475833333333333.
[I 2025-12-28 06:49:15,345] Trial 2 finished with value: 0.5285833333333333 and parameters: {'k': 28, 'model': 'random_forest', 'rf_n_estimators': 191, 'rf_max_depth': 16, 'rf_min_samples_split': 3, 'rf_min_samples_leaf': 5}. Best is trial 1 with value: 0.5475833333333333.
[I 2025-12-28 06:49:15,677] Trial 3 finished with value: 0.6027291666666666 and parameters: {'k': 80, 'mo

Best ROC-AUC: 0.6027291666666666
Best configuration:
{'k': 80, 'model': 'decision_tree', 'dt_max_depth': 6, 'dt_min_samples_split': 3, 'dt_min_samples_leaf': 5}


In [ ]:
best = study.best_params
model_name = best["model"]
k = best["k"]

if model_name == "svm":
    clf = SVC(
        C=best["svm_C"],
        gamma=best["svm_gamma"],
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=42
    )

elif model_name == "logistic":
    clf = LogisticRegression(
        C=best["logreg_C"],
        penalty="l2",
        solver="liblinear",
        class_weight="balanced",
        max_iter=2000,
        random_state=42
    )

elif model_name == "decision_tree":
    clf = DecisionTreeClassifier(
        max_depth=best["dt_max_depth"],
        min_samples_split=best["dt_min_samples_split"],
        min_samples_leaf=best["dt_min_samples_leaf"],
        class_weight="balanced",
        random_state=42
    )

else:
    clf = RandomForestClassifier(
        n_estimators=best["rf_n_estimators"],
        max_depth=best["rf_max_depth"],
        min_samples_split=best["rf_min_samples_split"],
        min_samples_leaf=best["rf_min_samples_leaf"],
        class_weight="balanced",
        n_jobs=-1,
        random_state=42
    )

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("selector", SelectKBest(score_func=f_classif, k=k)),
    ("clf", clf)
])

final_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('selector', SelectKBest(k=80)),
                ('clf',
                 DecisionTreeClassifier(class_weight='balanced', max_depth=6,
                                        min_samples_leaf=5, min_samples_split=3,
                                        random_state=42))])

In [ ]:
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

print("Test Accuracy:", accuracy_score(y_test, y_pred))
print("Test ROC AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Test Accuracy: 0.5365853658536586
Test ROC AUC: 0.47580645161290325

Classification Report:
                         precision    recall  f1-score   support

         adenocarcinoma       0.20      0.30      0.24        10
squamous cell carcinoma       0.73      0.61      0.67        31

               accuracy                           0.54        41
              macro avg       0.47      0.46      0.45        41
           weighted avg       0.60      0.54      0.56        41

Confusion Matrix:
[[ 3  7]
 [12 19]]


In [ ]:
selected_mask = best_model.named_steps["feature_selection"].get_support()
selected_features = X.columns[selected_mask]

print("Selected features:")
print(selected_features.tolist())


Selected features:
['original_shape_Elongation', 'original_shape_Flatness', 'original_shape_MajorAxisLength', 'original_shape_Maximum3DDiameter', 'original_shape_Sphericity', 'original_firstorder_90Percentile', 'original_firstorder_Entropy', 'original_firstorder_Maximum', 'original_firstorder_Mean', 'original_firstorder_Minimum', 'original_firstorder_Range', 'original_firstorder_RootMeanSquared', 'original_firstorder_Uniformity', 'original_glcm_Autocorrelation', 'original_glcm_JointAverage', 'original_glcm_ClusterProminence', 'original_glcm_ClusterShade', 'original_glcm_DifferenceVariance', 'original_glcm_JointEnergy', 'original_glcm_Idm', 'original_glcm_Id', 'original_glcm_Idn', 'original_glcm_InverseVariance', 'original_glcm_MaximumProbability', 'original_glrlm_GrayLevelNonUniformityNormalized', 'original_glrlm_HighGrayLevelRunEmphasis', 'original_glrlm_LongRunEmphasis', 'original_glrlm_LongRunHighGrayLevelEmphasis', 'original_glrlm_RunLengthNonUniformityNormalized', 'original_glrlm_